# Train frozen DINO + linear narration head

The official TaskData test set is not used for training, validation, class balancing, threshold selection, early stopping, or architecture decisions. Only labeled development data is used. OCR/GT fields are for supervision and auditing; model input is the padded image crop only. This notebook has not been executed.

Only the head trains. Validation retains its natural distribution. These validation results are model-selection scores, not an independent final-test estimate.

In [13]:
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel

ROOT = Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi/isNarratorText')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = torch.load(ROOT / 'narration_data.pt', map_location='cpu', weights_only=True)
assert data['settings']['data_source'] == 'development'
samples = data['samples']
train_sequences = {samples[i]['sequence_id'] for i in data['splits']['train']}
validation_sequences = {samples[i]['sequence_id'] for i in data['splits']['validation']}
assert train_sequences.isdisjoint(validation_sequences)
assert data['splits']['validation'] == [i for i, sample in enumerate(samples)
                                      if sample['sequence_id'] in validation_sequences]
assert set(data['train_sample_indices']) <= set(data['splits']['train'])


In [14]:
DINO_PATH = Path('C:/MODELS/dinov2-small')
SEED = data['settings']['seed']
BATCH_SIZE = 16
EPOCHS = 100
PATIENCE = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MIN_F1_IMPROVEMENT = 1e-4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
processor = AutoImageProcessor.from_pretrained(DINO_PATH, local_files_only=True)
encoder = AutoModel.from_pretrained(DINO_PATH, local_files_only=True)
FEATURE_DIM = encoder.config.hidden_size


In [15]:
class CropDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        sample = samples[self.indices[index]]
        with Image.open(sample['page_path']) as image:
            crop = image.convert('RGB').crop(tuple(sample['padded_bbox']))
        return crop, float(sample['label'])


def collate_crops(batch):
    crops, labels = zip(*batch)
    # Use DINO's own resize and normalization.
    pixels = processor(images=list(crops), return_tensors='pt')['pixel_values']
    return pixels, torch.tensor(labels, dtype=torch.float32)


class NarrationClassifier(nn.Module):
    def __init__(self, encoder, feature_dim):
        super().__init__()
        self.encoder = encoder.requires_grad_(False).eval()
        self.classifier = nn.Linear(feature_dim, 1)

    # Keep DINO in evaluation mode when the head trains.
    def train(self, mode=True):
        super().train(mode)
        self.encoder.eval()
        return self

    def forward(self, pixel_values):
        # Extract frozen CLS features without encoder gradients.
        with torch.no_grad():
            features = self.encoder(pixel_values=pixel_values).last_hidden_state[:, 0]
        return self.classifier(features).squeeze(-1)


def metrics_at_threshold(probabilities, labels, threshold):
    predictions = probabilities >= threshold
    positive = labels == 1
    tp = int((predictions & positive).sum())
    fp = int((predictions & ~positive).sum())
    tn = int((~predictions & ~positive).sum())
    fn = int((~predictions & positive).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    negative_recall = tn / max(tn + fp, 1)
    return dict(narration_precision=precision, narration_recall=recall,
                narration_f1=2 * tp / max(2 * tp + fp + fn, 1),
                narration_f0_5=1.25 * tp / max(1.25 * tp + fp + 0.25 * fn, 1),
                not_narration_recall=negative_recall,
                balanced_accuracy=(recall + negative_recall) / 2,
                accuracy=(tp + tn) / max(len(labels), 1), tp=tp, fp=fp, tn=tn, fn=fn)


def evaluate(loader):
    model.eval()
    probabilities, labels_all = [], []
    loss_total = 0.0
    with torch.no_grad():
        for pixels, labels in loader:
            logits = model(pixels.to(DEVICE))
            loss_total += loss_fn(logits, labels.to(DEVICE)).item() * len(labels)
            probabilities.append(logits.sigmoid().cpu())
            labels_all.append(labels)
    return torch.cat(probabilities).numpy(), torch.cat(labels_all).numpy(), loss_total / len(loader.dataset)


In [16]:
model = NarrationClassifier(encoder, FEATURE_DIM).to(DEVICE)
trainable = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
assert trainable == ['classifier.weight', 'classifier.bias']
assert all(not parameter.requires_grad for parameter in model.encoder.parameters())
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(name, parameter.numel())
positive_train = {i for i in data['splits']['train'] if samples[i]['label'] == 1}
assert positive_train <= set(data['train_sample_indices'])
assert sum(samples[i]['label'] == 0 for i in data['train_sample_indices']) <= 2 * len(positive_train)
train_loader = DataLoader(CropDataset(data['train_sample_indices']), batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate_crops, num_workers=0,
    generator=torch.Generator().manual_seed(SEED))
validation_loader = DataLoader(CropDataset(data['splits']['validation']), batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_crops, num_workers=0)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


classifier.weight 384
classifier.bias 1


## Select threshold on validation only

Search all distinct validation probabilities and endpoints. Cumulative counts make the exact search efficient. Threshold ties use balanced accuracy, then closeness to 0.5. Checkpoint F1 within 0.0001 is essentially tied; balanced accuracy breaks the tie. Patience resets only for meaningful F1 improvement.

In [10]:
# Search validation probabilities without touching official test data.
def select_threshold(probabilities, labels):
    order = np.argsort(-probabilities, kind='stable')
    sorted_probabilities = probabilities[order]
    sorted_labels = labels[order]
    ends = np.r_[np.flatnonzero(np.diff(sorted_probabilities) != 0), len(labels) - 1]
    tp = np.cumsum(sorted_labels)[ends]
    fp = ends + 1 - tp
    fn = labels.sum() - tp
    tn = (labels == 0).sum() - fp
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    ba = (tp / max(labels.sum(), 1) + tn / max((labels == 0).sum(), 1)) / 2
    best = max(range(len(ends)), key=lambda i: (f1[i], ba[i], -abs(sorted_probabilities[ends[i]] - 0.5)))
    thresholds = [0.0, float(sorted_probabilities[ends[best]]), 1.0]
    threshold = max(thresholds, key=lambda t: (
        metrics_at_threshold(probabilities, labels, t)['narration_f1'],
        metrics_at_threshold(probabilities, labels, t)['balanced_accuracy'], -abs(t - 0.5)))
    return threshold, metrics_at_threshold(probabilities, labels, threshold)


In [ ]:
best_f1 = -1.0
best_balanced_accuracy = -1.0
patience_f1 = -1.0
stale_epochs = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_total = 0.0
    for pixels, labels in train_loader:
        optimizer.zero_grad()
        logits = model(pixels.to(DEVICE))
        loss = loss_fn(logits, labels.to(DEVICE))
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item() * len(labels)
    probabilities, labels, validation_loss = evaluate(validation_loader)
    threshold, metrics = select_threshold(probabilities, labels)
    f1, ba, precision = metrics['narration_f1'], metrics['balanced_accuracy']
    improved = f1 > best_f1 + MIN_F1_IMPROVEMENT
    tied = abs(f1 - best_f1) <= MIN_F1_IMPROVEMENT
    if improved or (tied and ba > best_balanced_accuracy):
        best_f1, best_balanced_accuracy = f1, ba
        checkpoint = dict(
            classifier_state_dict={key: value.detach().cpu().clone()
                                   for key, value in model.classifier.state_dict().items()},
            encoder_path=str(DINO_PATH), feature_dim=FEATURE_DIM,
            validation_threshold=threshold, validation=metrics, epoch=epoch,
            settings=dict(**data['settings'], learning_rate=LEARNING_RATE,
                          weight_decay=WEIGHT_DECAY, patience=PATIENCE,
                          epochs=EPOCHS, batch_size=BATCH_SIZE,
                          min_f1_improvement=MIN_F1_IMPROVEMENT))
        # Replace the single best head checkpoint.
        torch.save(checkpoint, ROOT / 'narration_linear_head.pt')
    if f1 > patience_f1 + MIN_F1_IMPROVEMENT:
        patience_f1 = f1
        stale_epochs = 0
    else:
        stale_epochs += 1
    print(f'Epoch {epoch:02d} | train loss {train_loss_total / len(train_loader.dataset):.3f} | '
          f'val loss {validation_loss:.3f} | F1 {f1:.3f} | '
          f'precision {metrics["narration_precision"]:.3f} | recall {metrics["narration_recall"]:.3f} | '
          f'BA {ba:.3f} | threshold {threshold:.4f}')
    if stale_epochs >= PATIENCE:
        print('Early stopping.')
        break
print('Best head saved:', ROOT / 'narration_linear_head.pt')


Epoch 01 | train loss 0.761 | val loss 0.303 | F1 0.298 | precision 0.250 | recall 0.368 | BA 0.653 | threshold 0.3982
Epoch 02 | train loss 0.600 | val loss 0.398 | F1 0.491 | precision 0.382 | recall 0.684 | BA 0.811 | threshold 0.5881
Epoch 03 | train loss 0.479 | val loss 0.318 | F1 0.531 | precision 0.433 | recall 0.684 | BA 0.817 | threshold 0.5856
Epoch 04 | train loss 0.422 | val loss 0.301 | F1 0.556 | precision 0.429 | recall 0.789 | BA 0.865 | threshold 0.5697
Epoch 05 | train loss 0.357 | val loss 0.335 | F1 0.588 | precision 0.469 | recall 0.789 | BA 0.869 | threshold 0.6737
Epoch 06 | train loss 0.327 | val loss 0.259 | F1 0.615 | precision 0.485 | recall 0.842 | BA 0.896 | threshold 0.5717
Epoch 07 | train loss 0.311 | val loss 0.277 | F1 0.604 | precision 0.471 | recall 0.842 | BA 0.894 | threshold 0.6241
Epoch 08 | train loss 0.289 | val loss 0.290 | F1 0.596 | precision 0.500 | recall 0.737 | BA 0.847 | threshold 0.6937
Epoch 09 | train loss 0.277 | val loss 0.288 | F

## Alternative: high-precision narration gate

Run the setup, model, loader, and helper cells above, then run this collapsed cell instead of the F1 training loop. It starts a fresh linear head and optimizer.

Require natural validation narration precision >= `TARGET_PRECISION = 0.90`. Among eligible thresholds, maximize narration recall; break ties with precision, then balanced accuracy. Select checkpoints by the same rule. Only meaningful eligible recall improvement resets patience (15 epochs; maximum 100 epochs). Before any eligible threshold exists, continue up to the epoch limit rather than stopping on an infeasible objective.

Report precision, recall, F0.5, F1, and balanced accuracy. If no threshold reaches the precision target, report failure to meet the target and save no new checkpoint. Any existing checkpoint is left unchanged in that case. Otherwise save the single best head to `narration_linear_head.pt`, replacing any previous head. The validation notebook uses the saved threshold without tuning.

The target is measured on development validation and is not a guarantee on unseen data. All accepted validation samples remain included; only training negatives are downsampled. The official test set remains untouched.


In [17]:
# Require high validation precision, then maximize narration recall.
TARGET_PRECISION = 0.90
MIN_RECALL_IMPROVEMENT = 1e-4
assert 0 < TARGET_PRECISION <= 1


def select_constrained_threshold(probabilities, labels):
    order = np.argsort(-probabilities, kind='stable')
    sorted_probabilities = probabilities[order]
    sorted_labels = labels[order]
    ends = np.r_[np.flatnonzero(np.diff(sorted_probabilities) != 0), len(labels) - 1]
    tp = np.cumsum(sorted_labels)[ends]
    fp = ends + 1 - tp
    tn = (labels == 0).sum() - fp
    precision = tp / (ends + 1)
    recall = tp / max(labels.sum(), 1)
    ba = (recall + tn / max((labels == 0).sum(), 1)) / 2
    eligible = np.flatnonzero(precision >= TARGET_PRECISION)
    if len(eligible):
        best = max(eligible, key=lambda i: (recall[i], precision[i], ba[i]))
    else:
        # Return diagnostic metrics without treating this threshold as eligible.
        best = max(range(len(ends)), key=lambda i: (precision[i], recall[i], ba[i]))
    threshold = float(sorted_probabilities[ends[best]])
    metrics = metrics_at_threshold(probabilities, labels, threshold)
    return threshold, metrics, bool(len(eligible))


# Reset the head and optimizer for this alternative experiment.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
train_loader.generator.manual_seed(SEED)
model = NarrationClassifier(encoder, FEATURE_DIM).to(DEVICE)
assert all(not parameter.requires_grad for parameter in model.encoder.parameters())
assert [name for name, parameter in model.named_parameters() if parameter.requires_grad] == [
    'classifier.weight', 'classifier.bias']
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
loss_fn = nn.BCEWithLogitsLoss()

best_key = None
patience_recall = -1.0
stale_epochs = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_total = 0.0
    for pixels, labels in train_loader:
        optimizer.zero_grad()
        logits = model(pixels.to(DEVICE))
        loss = loss_fn(logits, labels.to(DEVICE))
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item() * len(labels)

    # Select only from probabilities on the complete natural validation split.
    probabilities, labels, validation_loss = evaluate(validation_loader)
    threshold, metrics, eligible = select_constrained_threshold(probabilities, labels)
    precision = metrics['narration_precision']
    recall = metrics['narration_recall']
    ba = metrics['balanced_accuracy']
    key = (recall, precision, ba)
    if eligible and (best_key is None or key > best_key):
        best_key = key
        checkpoint = dict(
            classifier_state_dict={key: value.detach().cpu().clone()
                                   for key, value in model.classifier.state_dict().items()},
            encoder_path=str(DINO_PATH), feature_dim=FEATURE_DIM,
            validation_threshold=threshold, validation=metrics, epoch=epoch,
            selection_metric='recall_at_target_precision',
            settings=dict(**data['settings'], learning_rate=LEARNING_RATE,
                          weight_decay=WEIGHT_DECAY, patience=PATIENCE,
                          epochs=EPOCHS, batch_size=BATCH_SIZE,
                          target_precision=TARGET_PRECISION,
                          min_recall_improvement=MIN_RECALL_IMPROVEMENT))
        torch.save(checkpoint, ROOT / 'narration_linear_head.pt')

    # Start patience only after the precision constraint has been achieved.
    if eligible and recall > patience_recall + MIN_RECALL_IMPROVEMENT:
        patience_recall = recall
        stale_epochs = 0
    elif best_key is not None:
        stale_epochs += 1
    print(f'Epoch {epoch:02d} | train loss {train_loss_total / len(train_loader.dataset):.3f} | '
          f'val loss {validation_loss:.3f} | precision {precision:.3f} | recall {recall:.3f} | '
          f'F0.5 {metrics["narration_f0_5"]:.3f} | F1 {metrics["narration_f1"]:.3f} | '
          f'BA {ba:.3f} | threshold {threshold:.4f} | target met {eligible} | '
          f'patience {stale_epochs}/{PATIENCE}')
    if best_key is not None and stale_epochs >= PATIENCE:
        print('Early stopping: recall at the precision target stopped improving.')
        break
if best_key is None:
    print(f'No validation threshold met precision >= {TARGET_PRECISION:.2f}; no new head saved.')
else:
    print(f'Best eligible recall: {best_key[0]:.4f} | precision: {best_key[1]:.4f}')
    print('Best high-precision head saved:', ROOT / 'narration_linear_head.pt')


Epoch 01 | train loss 0.608 | val loss 0.286 | precision 0.500 | recall 0.158 | F0.5 0.349 | F1 0.240 | BA 0.574 | threshold 0.5645 | target met False | patience 0/20
Epoch 02 | train loss 0.504 | val loss 0.466 | precision 0.667 | recall 0.105 | F0.5 0.323 | F1 0.182 | BA 0.551 | threshold 0.9162 | target met False | patience 0/20
Epoch 03 | train loss 0.414 | val loss 0.261 | precision 1.000 | recall 0.105 | F0.5 0.370 | F1 0.190 | BA 0.553 | threshold 0.8942 | target met True | patience 0/20
Epoch 04 | train loss 0.381 | val loss 0.343 | precision 1.000 | recall 0.105 | F0.5 0.370 | F1 0.190 | BA 0.553 | threshold 0.9579 | target met True | patience 1/20
Epoch 05 | train loss 0.337 | val loss 0.283 | precision 1.000 | recall 0.105 | F0.5 0.370 | F1 0.190 | BA 0.553 | threshold 0.9583 | target met True | patience 2/20
Epoch 06 | train loss 0.313 | val loss 0.267 | precision 1.000 | recall 0.105 | F0.5 0.370 | F1 0.190 | BA 0.553 | threshold 0.9651 | target met True | patience 3/20


KeyboardInterrupt: 